# Imports

In [22]:
import time
import cv2
import numpy as np
import pandas as pd

# 1. Make three mock videos
- Video 1 & 2: Similar (same position, different colors)
- Video 3: Different (different position, different color)

In [23]:
def create_video(output_path, circle_y, circle_color, width=320, height=240, fps=10, duration=2):
    """Create a video of a circle moving left to right."""
    total_frames = fps * duration
    radius = 20
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    for i in range(total_frames):
        frame = np.ones((height, width, 3), dtype=np.uint8) * 255  # white background
        x = int(radius + (width - 2*radius) * (i / (total_frames - 1)))
        cv2.circle(frame, (x, circle_y), radius, circle_color, -1)
        out.write(frame)
    out.release()

# Create videos
create_video("video1.mp4", circle_y=120, circle_color=(255, 0, 0))   # Blue, center
create_video("video2.mp4", circle_y=120, circle_color=(0, 0, 255))   # Red, center (similar to video1)
create_video("video3.mp4", circle_y=200, circle_color=(0, 255, 0))   # Green, bottom (different)

# 2. Convert to tensors and compare similarities

In [28]:
def load_video_tensor(path):
    """Load video as numpy tensor (frames, height, width, channels)."""
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return np.array(frames, dtype=np.float32) / 255.0

# Load tensors
tensor1 = load_video_tensor("video1.mp4")
tensor2 = load_video_tensor("video2.mp4")
tensor3 = load_video_tensor("video3.mp4")

def cosine_similarity(t1, t2):
    """Cosine similarity between flattened tensors."""
    a, b = t1.flatten(), t2.flatten()
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare original tensors
print("\nOriginal Tensor Comparisons:")
print("-" * 60)

pairs = [("Tensor1 vs Tensor2", tensor1, tensor2),
         ("Tensor2 vs Tensor3", tensor2, tensor3),
         ("Tensor1 vs Tensor3", tensor1, tensor3)]

original_results = []
for name, t1, t2 in pairs:
    start = time.time()
    cos = cosine_similarity(t1, t2)
    elapsed = time.time() - start
    original_results.append((name, cos, elapsed))
    print(f"{name}: Cosine={round(cos,4)}, Time={round(elapsed*1000,2)}ms")


Original Tensor Comparisons:
------------------------------------------------------------
Tensor1 vs Tensor2: Cosine=0.9947999715805054, Time=45.24ms
Tensor2 vs Tensor3: Cosine=0.989300012588501, Time=40.91ms
Tensor1 vs Tensor3: Cosine=0.9891999959945679, Time=36.73ms


# 3. Apply tensor decomposition and compare

In [25]:
def svd_decomposition(tensor, rank=20):
    """Truncated SVD decomposition."""
    unfolded = tensor.reshape(tensor.shape[0], -1)
    U, S, Vt = np.linalg.svd(unfolded, full_matrices=False)
    return U[:, :rank], S[:rank], Vt[:rank, :]

def tucker_decomposition(tensor, ranks=(5, 10, 10, 3)):
    """Tucker/HOSVD decomposition - returns factor matrices for each mode."""
    factors = []
    for mode in range(tensor.ndim):
        unfolded = np.moveaxis(tensor, mode, 0).reshape(tensor.shape[mode], -1)
        U, S, Vt = np.linalg.svd(unfolded, full_matrices=False)
        factors.append(U[:, :min(ranks[mode], U.shape[1])])
    return factors

def compare_svd(decomp1, decomp2):
    """Compare two SVD decompositions."""
    U1, S1, _ = decomp1
    U2, S2, _ = decomp2
    r1 = (U1 @ np.diag(S1)).flatten()
    r2 = (U2 @ np.diag(S2)).flatten()
    return np.dot(r1, r2) / (np.linalg.norm(r1) * np.linalg.norm(r2))

def compare_tucker(factors1, factors2):
    """Compare two Tucker decompositions."""
    sims = []
    for f1, f2 in zip(factors1, factors2):
        k = min(f1.shape[1], f2.shape[1])
        a, b = f1[:, :k].flatten(), f2[:, :k].flatten()
        sims.append(abs(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))))
    return np.mean(sims)

# Perform decompositions
print("\nTensor Decomposition:")
print("-" * 60)

# SVD
start = time.time()
svd1 = svd_decomposition(tensor1)
svd2 = svd_decomposition(tensor2)
svd3 = svd_decomposition(tensor3)
svd_decomp_time = time.time() - start
print(f"SVD decomposition time: {round(svd_decomp_time*1000,2)}ms")

# Tucker
start = time.time()
tucker1 = tucker_decomposition(tensor1)
tucker2 = tucker_decomposition(tensor2)
tucker3 = tucker_decomposition(tensor3)
tucker_decomp_time = time.time() - start
print(f"Tucker decomposition time: {round(tucker_decomp_time*1000,2)}ms")

# Compare decomposed tensors
print("\nDecomposed Tensor Comparisons:")
print("-" * 60)

svd_pairs = [("Tensor1 vs Tensor2", svd1, svd2),
             ("Tensor2 vs Tensor3", svd2, svd3),
             ("Tensor1 vs Tensor3", svd1, svd3)]

tucker_pairs = [("Tensor1 vs Tensor2", tucker1, tucker2),
                ("Tensor2 vs Tensor3", tucker2, tucker3),
                ("Tensor1 vs Tensor3", tucker1, tucker3)]

svd_results = []
tucker_results = []

for (name, s1, s2), (_, t1, t2) in zip(svd_pairs, tucker_pairs):
    start = time.time()
    svd_sim = compare_svd(s1, s2)
    svd_time = time.time() - start

    start = time.time()
    tucker_sim = compare_tucker(t1, t2)
    tucker_time = time.time() - start

    svd_results.append((name, svd_sim, svd_time))
    tucker_results.append((name, tucker_sim, tucker_time))

    print(f"{name}:")
    print(f"  SVD: {round(svd_sim,4)} ({round(svd_time*1000,4)}ms)")
    print(f"  Tucker: {round(tucker_sim,4)} ({round(tucker_time*1000,4)}ms)")


Tensor Decomposition:
------------------------------------------------------------
SVD decomposition time: 1288.82ms
Tucker decomposition time: 12918.38ms

Decomposed Tensor Comparisons:
------------------------------------------------------------
Tensor1 vs Tensor2:
  SVD: 0.9979000091552734 (0.1292ms)
  Tucker: 0.7502999901771545 (0.4611ms)
Tensor2 vs Tensor3:
  SVD: 0.9984999895095825 (0.0656ms)
  Tucker: 0.6876999735832214 (0.1609ms)
Tensor1 vs Tensor3:
  SVD: 0.9994999766349792 (0.0446ms)
  Tucker: 0.6869000196456909 (0.1848ms)


# Summary

In [26]:
summary_sim = pd.DataFrame(columns=['Tensor Pair', 'Cosine Similarity', 'SVD Similarity', 'Tucker Similarity'])
summary_sim.set_index('Tensor Pair', inplace=True)
for i in range(3):
    name = original_results[i][0]
    summary_sim.at[name, 'Cosine Similarity'] = original_results[i][1]
    summary_sim.at[name, 'SVD Similarity'] = svd_results[i][1]
    summary_sim.at[name, 'Tucker Similarity'] = tucker_results[i][1]

summary_sim

,Cosine Similarity,SVD Similarity,Tucker Similarity
Tensor Pair,,,
Tensor1 vs Tensor2,0.9948,0.997929,0.750343
Tensor2 vs Tensor3,0.989259,0.998462,0.687656
Tensor1 vs Tensor3,0.989153,0.99955,0.68694


In [27]:
print("\nTiming Summary:")
print(f"  Original comparison (total): {round(sum(r[2] for r in original_results)*1000,2)}ms")
print(f"  SVD decomposition: {round(svd_decomp_time*1000,2)}ms")
print(f"  SVD comparison (total): {round(sum(r[2] for r in svd_results)*1000,4)}ms")
print(f"  Tucker decomposition: {round(tucker_decomp_time*1000,2)}ms")
print(f"  Tucker comparison (total): {round(sum(r[2] for r in tucker_results)*1000,4)}ms")


Timing Summary:
  Original comparison (total): 40.99ms
  SVD decomposition: 1288.82ms
  SVD comparison (total): 0.2394ms
  Tucker decomposition: 12918.38ms
  Tucker comparison (total): 0.8068ms
